In [18]:
import math
import torch
from einops import einsum, rearrange

# Test Linear

In [2]:
from modules import LinearLayer

linear1 = LinearLayer(in_features=4, out_features=3)
linear1.W.shape

torch.Size([3, 4])

In [3]:
x1d = torch.randn(4)           # (d_in)
x2d = torch.randn(5, 4)        # (batch, d_in)
x3d = torch.randn(2, 5, 4)     # (batch, seq, d_in)

print(linear1(x1d).shape)
print(linear1(x2d).shape)
print(linear1(x3d).shape)

torch.Size([3])
torch.Size([5, 3])
torch.Size([2, 5, 3])


# Test Embedding

In [4]:
from modules import EmbeddingLayer

emb = EmbeddingLayer(10, 4)  # vocab=10, d_model=4

x0 = torch.tensor(3)                     # scalar
x1 = torch.tensor([1, 2, 3])              # (seq_len,)
x2 = torch.tensor([[1, 2], [3, 4]])       # (batch, seq_len)

print(emb(x0).shape, emb(x0).grad_fn)
print(emb(x1).shape, emb(x1).grad_fn)
print(emb(x2).shape, emb(x2).grad_fn)

torch.Size([4]) <SelectBackward0 object at 0x106076f80>
torch.Size([3, 4]) <IndexBackward0 object at 0x1475f1ba0>
torch.Size([2, 2, 4]) <IndexBackward0 object at 0x106076f80>


In [5]:
emb(x2).grad_fn

# Test RMSNorm

In [6]:
from modules import RMSNormLayer

norm = RMSNormLayer(4)
x = torch.randn(2, 3, 4)  # (batch, seq_len, d_model)
out = norm(x)

print(out.shape)  # should be (2, 3, 4) — same as input
print(out.dtype)  # should match x.dtype
print(x.dtype)

torch.Size([2, 3, 4])
torch.float32
torch.float32


# Test SwiGLU

In [7]:
from modules import SwiGLU

swiglu = SwiGLU(d_model=8, d_ff=24)
x = torch.randn(2, 5, 8)
out = swiglu(x)

print(out.shape)  # what do you expect, and what do you get?

torch.Size([2, 5, 8])


In [8]:
buf = torch.randn(10, 4)  # (max_seq_len, d_k//2)
pos = torch.tensor([0, 1, 2])  # (seq_len,)
print(buf[pos].shape)  # what do you get?

pos_batched = torch.tensor([[0,1,2],[3,4,5]])  # (batch, seq_len)
print(buf[pos_batched].shape)  # what do you get?

torch.Size([3, 4])
torch.Size([2, 3, 4])


# Test RoPE

In [9]:
from modules import RoPE

# Test Softmax

In [10]:
from modules import SoftmaxLayer

x = torch.randn(size=(2, 2, 4))
SoftmaxLayer(x, dim=2)

tensor([[[0.7571, 0.1016, 0.0417, 0.0996],
         [0.0640, 0.7094, 0.1669, 0.0597]],

        [[0.0395, 0.4289, 0.0370, 0.4946],
         [0.0487, 0.4153, 0.2745, 0.2615]]])

# Test Attention

In [11]:
from modules import AttentionLayer

In [12]:
Q = torch.randn(2, 3, 2)
K = torch.randn(2, 4, 2)
V = torch.randn(4,2)
mask = torch.randint(high=2, size=(3,4))
mask = torch.where(mask == 1, True, False)
output = AttentionLayer(Q, K, V, mask)
output

tensor([[[ 0.2970,  0.4867],
         [-0.0631,  0.2325],
         [ 0.3846,  0.4580]],

        [[ 0.1900,  0.4811],
         [-0.0631,  0.2325],
         [ 0.1911,  0.4580]]])

# Debug

In [42]:
import torch.nn as nn
from modules import RoPE

In [30]:
d_model = 1600
num_heads = 25
max_seq_len = 100

h = num_heads
d_k = d_model // h
d_v = d_model // h

In [31]:
Query = LinearLayer(in_features=d_model, out_features=d_k * h)
Key = LinearLayer(in_features=d_model, out_features=d_k * h)
Value = LinearLayer(in_features=d_model, out_features=d_v * h)

O = LinearLayer(in_features=d_v * h, out_features=d_model)

Query.W.shape, Key.W.shape, Value.W.shape, O.W.shape

(torch.Size([1600, 1600]),
 torch.Size([1600, 1600]),
 torch.Size([1600, 1600]),
 torch.Size([1600, 1600]))

In [32]:
x = torch.randn(size=(100, 1600))
x.shape

torch.Size([100, 1600])

In [33]:
# mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
# mask.shape

In [34]:
# Q_out = Query(x)
# K_out = Key(x)
# V_out = Value(x)

# Q_out.shape, K_out.shape, V_out.shape

In [35]:
# Q_out = rearrange(Q_out, " ... seq (heads d_k) -> ... heads seq d_k", heads=h)
# K_out = rearrange(K_out, " ... seq (heads d_k) -> ... heads seq d_k", heads=h)
# V_out = rearrange(V_out, " ... seq (heads d_v) -> ... heads seq d_v", heads=h)

# Q_out.shape, K_out.shape, V_out.shape

In [36]:
# from modules import AttentionLayer

In [37]:
# multihead_attention = AttentionLayer(Q_out, K_out, V_out, mask=mask)
# multihead_attention.shape

In [38]:
# temp = rearrange(multihead_attention, " ... heads seq d_v -> ... seq (heads d_v)", heads=h)
# temp.shape

In [39]:
# O(temp).shape

In [48]:
class MultiHeadSelfAttentionLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int, theta: float = None, device=None, dtype=None):
        super(MultiHeadSelfAttentionLayer, self).__init__()
        self.d_model = d_model
        self.h = num_heads
        self.d_k = self.d_model // self.h
        self.d_v = self.d_model // self.h
        self.max_seq_len = max_seq_len
        self.device = device

        self.Query = LinearLayer(in_features=self.d_model, out_features=self.d_k * self.h, device=device, dtype=dtype)
        self.Key = LinearLayer(in_features=self.d_model, out_features=self.d_k * self.h, device=device, dtype=dtype)
        self.Value = LinearLayer(in_features=self.d_model, out_features=self.d_v * self.h, device=device, dtype=dtype)

        self.Output = LinearLayer(in_features=self.d_v * self.h, out_features=self.d_model, device=device, dtype=dtype)

        self.rope = RoPE(theta=theta, d_k=self.d_k, max_seq_len=self.max_seq_len, device=device)

    def forward(self, x, token_positions=None):
        Q_out = self.Query(x)
        K_out = self.Key(x)
        V_out = self.Value(x)
        Q_out = rearrange(Q_out, " ... seq (heads d_k) -> ... heads seq d_k", heads=self.h)
        K_out = rearrange(K_out, " ... seq (heads d_k) -> ... heads seq d_k", heads=self.h)
        V_out = rearrange(V_out, " ... seq (heads d_v) -> ... heads seq d_v", heads=self.h)

        if token_positions is not None:
            print('asd')
            Q_out = self.rope(Q_out, token_positions)
            K_out = self.rope(K_out, token_positions)
        
        mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len, dtype=torch.bool, device=self.device))
        multihead_attention = AttentionLayer(Q_out, K_out, V_out, mask=mask)
        multihead_attention = rearrange(multihead_attention, " ... heads seq d_v -> ... seq (heads d_v)", heads=self.h)
        return self.Output(multihead_attention)

temp = MultiHeadSelfAttentionLayer(d_model, num_heads, max_seq_len, theta=1000)

In [51]:
temp(x, 10)

asd


tensor([[-0.8477,  0.6207,  1.3778,  ...,  1.0733, -1.8537, -0.4284],
        [ 0.3134, -0.1309,  0.7840,  ...,  0.7571, -0.8123, -0.9993],
        [ 1.2130,  0.1637,  0.0026,  ...,  0.3755, -0.6998, -0.8293],
        ...,
        [ 0.2350,  0.0171, -0.1006,  ..., -0.3224,  0.0857,  0.1692],
        [ 0.1142, -0.0421,  0.1367,  ..., -0.3323,  0.0842,  0.2097],
        [ 0.1894, -0.2644,  0.2742,  ..., -0.3507,  0.0079,  0.0245]],
       grad_fn=<ViewBackward0>)